In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [ ]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

# fastwave binary (use 3d version if it exists, otherwise main build)
fastwave_bin = str(rave_sim_dir / 'fast-wave' / 'build-Release' / 'fastwave')
if not os.path.exists(fastwave_bin):
    fastwave_bin = str(rave_sim_dir / 'fast-wave - 260428 3d版本' / 'build-Release' / 'fastwave')
print(f'fastwave binary: {fastwave_bin}')
print(f'  exists: {os.path.exists(fastwave_bin)}')

In [ ]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
import multisim
import config
import util
import propagation

## 1D Simulation via fastwave (C++ GPU)

Standard capsule_test pattern:
  config_dict → multisim.setup_simulation() → fastwave C++ → load → plot

This notebook uses a **neutral material sample** (no plasma) because
the C++ binary does not yet support the `plasma_sample` element type.

In [ ]:
# ---- generate a simple material grid ----
# A 2D (z x) grid with a W strip in vacuum

nz, nx = 3, 256
grid = np.zeros((nz, nx), dtype=np.uint32)
grid[:, 96:160] = 1   # W strip in the centre

print(f'Grid shape: {grid.shape}, W fill fraction: {grid.mean():.3f}')

grid_dir = Path.cwd() / 'fw_grids'
grid_dir.mkdir(exist_ok=True)
np.save(grid_dir / 'strip_grid.npy', grid)
print(f'Grid saved to {grid_dir}/strip_grid.npy')

In [ ]:
# ---- simulation config ----
energy = 8000.0  # eV
wl = propagation.convert_energy_wavelength(energy)
N = 2**18
dx = propagation.max_dx(0.01, 0.0, N, wl)

config_dict = {
    "sim_params": {
        "N": N,
        "dx": dx,
        "z_detector": 0.15,
        "detector_size": nx * dx,
        "detector_pixel_size_x": dx * 4,
        "detector_pixel_size_y": 1.0,
        "chunk_size": 4096,
    },
    "use_disk_vector": False,
    "save_final_u_vectors": False,
    "dtype": "c8",
    "multisource": {
        "type": "points",
        "energy_range": [7950, 8050],
        "x_range": [0, 0],
        "z": 0.0,
        "nr_source_points": 1,
        "seed": 1,
    },
    "elements": [
        {
            "type": "sample",
            "z_start": 0.01,
            "pixel_size_x": dx,
            "pixel_size_z": 1.0e-6,
            "grid_path": "strip_grid.npy",
            "materials": [["W", 19.35]],
            "x_positions": [0.0],
        },
    ],
}

print(f'N={N}, dx={dx:.3e} m')
sim_path = multisim.setup_simulation(config_dict, grid_dir, simulations_dir)
print(f'Simulation directory:')
print(f'  {sim_path}')

### Run fastwave (C++ GPU)

This calls the compiled C++ binary directly, exactly like the capsule_test notebooks.

In [ ]:
nr_sources = config_dict['multisource']['nr_source_points']

for i in tqdm(range(nr_sources)):
    cmd = f'CUDA_VISIBLE_DEVICES=0 {fastwave_bin} -s {i} {sim_path}'
    ret = os.system(cmd)
    if ret != 0:
        print(f'  fastwave returned {ret} for source {i}')
    
print('All sources done')

In [ ]:
# ---- load and plot results ----

wavefronts = util.load_wavefronts_filtered(sim_path)
print(f'Sources loaded: {len(wavefronts)}')

wf = np.sum([r[0] for r in wavefronts], axis=0)
print(f'Wavefront shape: {wf.shape}')
det = wf[0]

detector_x = util.detector_x_vector(
    config_dict['sim_params']['detector_size'],
    config_dict['sim_params']['detector_pixel_size_x'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Standard plot (cf. plt.plot(wf[0]) in capsule_test)
ax1.plot(detector_x * 1e3, det)
ax1.set_xlabel('x (mm)'); ax1.set_ylabel('Intensity')
ax1.set_title('Detector intensity (W strip sample)')
ax1.grid(True, alpha=0.3)

# Free-space reference: run vacuum sim
cfg_vac = config_dict.copy()
cfg_vac['elements'] = []
sp_vac = multisim.setup_simulation(cfg_vac, grid_dir, simulations_dir)
os.system(f'CUDA_VISIBLE_DEVICES=0 {fastwave_bin} -s 0 {sp_vac}')
wfs_vac = util.load_wavefronts_filtered(sp_vac)
det_vac = np.sum([r[0] for r in wfs_vac], axis=0)[0]

ax2.plot(detector_x * 1e3, det, label='With W strip')
ax2.plot(detector_x * 1e3, det_vac, '--', label='Free space', alpha=0.8)
ax2.set_xlabel('x (mm)'); ax2.set_ylabel('Intensity')
ax2.set_title('Comparison with free space')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fastwave_1d_result.png', dpi=150)
plt.show()

In [ ]:
from contextlib import redirect_stdout
with open('output_fastwave_1d.txt', 'w') as f:
    with redirect_stdout(f):
        for v in det:
            print(v)
print(f'Saved: output_fastwave_1d.txt ({len(det)} values)')

In [ ]:
print('=' * 44)
print('  fastwave 1D Simulation Complete')
print('=' * 44)
print(f'  sim_path: {sim_path}')
print(f'  detected shape: {det.shape}')
print(f'  GPU solver: {fastwave_bin}')
print()
print('  Note: fastwave C++ does not yet support')
print('  type="plasma_sample". This notebook uses')
print('  type="sample" (neutral materials) instead.')